In [ ]:
from pathlib import Path
import json

weather_records = []
source_dir = Path('data')
json_files = sorted(source_dir.glob('*.json'))

for json_file in json_files:
    with json_file.open('r', encoding='utf-8') as f:
        data = json.load(f)
    record = {
        "city": data["name"],
        "country": data["sys"]["country"],
        "temperature": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "pressure": data["main"]["pressure"],
        "weather": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"],
        "timestamp": data["dt"]
    }
    weather_records.append(record)
    print(f"Loaded: {record['city']} from {json_file.name}")

if not weather_records:
    raise RuntimeError("No local weather data found in the data/ folder. Add JSON files or verify the path.")

print(f"Total cities loaded: {len(weather_records)}")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, round, current_timestamp

spark = SparkSession.builder.appName('WeatherETL').getOrCreate()

df = spark.createDataFrame(weather_records)

print('✅ Raw Weather DataFrame:')
df.show()

weather_df = df.withColumn('temperature', round(col('temperature'), 1)) \
    .withColumn('feels_like', round(col('feels_like'), 1)) \
    .withColumn('wind_speed', round(col('wind_speed'), 1)) \
    .withColumn('datetime', from_unixtime(col('timestamp'))) \
    .withColumn('load_timestamp', current_timestamp()) \
    .drop('timestamp')

print('✅ Transformed Weather DataFrame:')
weather_df.show()
print(f'Total rows: {weather_df.count()}')


In [ ]:
weather_df.write.format('delta').mode('overwrite').saveAsTable('gold_weather_uk')

print('✅ Delta Table saved: gold_weather_uk')

spark.sql('SELECT city, country, temperature, weather, wind_speed, datetime FROM gold_weather_uk ORDER BY temperature DESC').show()
print('✅ Gold Weather Table verified!')
